# 01_Theory: BPE vs WordPiece 깊이있게 이해하기

> **핵심 목표:** 이 노트북은 README의 배경을 깔고 있다고 가정합니다.
> 여기서는 **실습에서 직접 사용할 2가지 모델의 개념, 기술, 작동원리**에 집중합니다.
> 
> 📖 배경/비용 차이는 README를 참고하세요.
> 🔧 이 노트북: 모델 자체의 이해

---

## 🔧 실습 모델 미리보기

이 노트북에서 배울 2가지 모델:

1. **GPT-2 (BPE)** ← 가장 대표적, 가장 간단
2. **mBERT (WordPiece)** ← 확률 기반, 의미 중심

각 모델을 02_practice.ipynb에서 코드로 실행해봅니다.

---

## 1️⃣ GPT-2 (BPE): 빈도 기반 토크나이제이션

### 핵심 아이디어

**"데이터에서 가장 자주 나타나는 바이트/문자 쌍을 찾아서 계속 병합한다"**

---

### 1.1 BPE의 탄생 배경

왜 BPE를 만들었을까?

```
문제 1: 모든 문자를 별개 토큰으로?
→ "hello" = 5개 토큰 (너무 많음)

문제 2: 모든 단어를 별개 토큰으로?
→ run, runs, running, ran = 4개 어휘 필요 (너무 많음)

해결책: 자주 나오는 패턴을 미리 합쳐놓자!
→ "ll"이 자주 나오면 하나의 토큰으로 만들기
→ "ing"이 자주 나오면 하나의 토큰으로 만들기
```

### 1.2 BPE의 알고리즘 (단계별)

#### Step 0: 초기 어휘 (글자 레벨)
```
말뭉치에 나오는 모든 글자를 개별 토큰으로 설정
vocab = {'h', 'e', 'l', 'o', 'w', 'r', 'd', ...}
```

#### Step 1: 가장 빈번한 쌍 찾기
```
"hello world" → ['h','e','l','l','o',' ','w','o','r','l','d']

인접한 쌍들:
('h','e'): 빈도 5
('e','l'): 빈도 3
('l','l'): 빈도 12  ← 가장 빈번!
('o',' '): 빈도 8
(' ','w'): 빈도 4
...

→ ('l','l')을 병합!
```

#### Step 2: 병합 수행
```
Before: ['h','e','l','l','o',' ','w','o','r','l','d']
After:  ['h','e','ll','o',' ','w','o','r','l','d']

vocab에 'll' 추가
```

#### Step 3: 반복
```
다시 가장 빈번한 쌍을 찾음: ('e','ll')
병합: ['h','ell','o',' ','w','o','r','l','d']

다시: ('w','o')
병합: ['h','ell','o',' ','wo','r','l','d']

...

목표 어휘 크기(~50K)에 도달할 때까지 반복
```

### 1.3 BPE의 핵심 특징

#### 특징 1: 탐욕적(Greedy) 선택
```
"최고의 선택"이 아니라 "지금 가장 자주 나오는" 쌍을 선택
→ 최적이 아닐 수도 있지만, 매우 빠름
```

#### 특징 2: 의미를 무시
```
빈도만 고려하므로 언어학적 의미는 상관없음
"ing"(동명사) vs "ing"(단순 문자 조합)
→ BPE는 구분 안 함, 둘 다 빈도로 판단
```

#### 특징 3: 공백 처리
```
단어 경계를 유지하기 위해 공백을 특수 문자로 치환
"hello world" → "hello<EOS>world"
또는 "hello </w>world"
또는 Llama처럼: "hello_ world" (밑줄 사용)

GPT-2의 경우: "hello" + "Ġworld" (Ġ = space)
```

### 1.4 BPE 어휘 구성

```
GPT-2 어휘 크기: ~50,000개

구성:
- 기본 문자: 256개 (모든 바이트)
- 병합된 문자 쌍: 49,744개

예시 어휘:
{'a', 'b', ..., 'z'} (26개)
{'ab', 'the', 'ing', 'er', ...} (패턴)
{'hello', 'world', 'Ġthe', 'Ġis'} (자주 나오는 완전 단어도)
```

---

## 2️⃣ mBERT (WordPiece): 확률 기반 토크나이제이션

### 핵심 아이디어

**"단순 빈도가 아니라, 두 토큰을 병합했을 때 확률이 가장 높아지는 조합을 선택한다"**

---

### 2.1 WordPiece vs BPE의 차이

```
BPE:      빈도 중심
          "이 쌍이 가장 자주 나타난다"

WordPiece: 확률 중심
          "이 쌍을 병합하면 모델의 가능도(Likelihood)가 가장 높아진다"
```

### 2.2 WordPiece 알고리즘

#### 핵심 개념: 정보 획득량 (Information Gain)

```
두 토큰 A와 B를 병합할 때의 정보 획득량:

InfoGain(A,B) = log(P(AB) / (P(A) × P(B)))

즉:
- P(AB) = 병합된 토큰의 확률
- P(A) × P(B) = 개별 토큰 확률의 곱

InfoGain이 크다 = 이 두 토큰은 자주 함께 나타난다
              = 합치면 더 의미있는 단위가 된다
```

#### 단계별 예시

```
Step 1: 단어를 공백으로 분할 (사전 토크나이저 필요!)
        "playing" → ['p', 'l', 'a', 'y', 'i', 'n', 'g']

Step 2: 각 인접 쌍의 정보 획득량 계산
        ('p','l'): InfoGain = 0.5
        ('l','a'): InfoGain = 0.3
        ('a','y'): InfoGain = 0.1
        ('y','i'): InfoGain = 0.2
        ('i','n'): InfoGain = 2.1  ← 가장 높음!
        ('n','g'): InfoGain = 0.8

Step 3: ('i','n') 병합
        ['p', 'l', 'a', 'y', 'in', 'g']

Step 4: 반복...
        다음: ('in','g')
        ['p', 'l', 'a', 'y', 'ing']
        
        다음: ('y','ing')
        ['p', 'l', 'a', 'ying']
        
        다음: ('a','ying')
        ['p', 'l', 'aying']
        
        ...
```

### 2.3 WordPiece의 핵심 특징

#### 특징 1: 사전 기반 (Pre-tokenizer 필수)
```
WordPiece는 반드시 단어 단위로 먼저 분할해야 함
→ 띄어쓰기가 있는 언어에만 적합
→ 중국어, 일본어 같은 띄어쓰기 없는 언어는 별도 처리 필요
```

#### 특징 2: ## 접두사 (서브워드 표시)
```
단어 내부의 서브워드에는 ##을 붙여서 표시

예: "playing"
결과: ['play', '##ing']
      (##ing은 "이전 토큰의 연속"을 의미)

이렇게 하면 원본 단어 복원 가능:
'play' + '##ing' → 'playing'
```

#### 특징 3: 확률 기반 선택
```
BPE의 "단순 빈도"보다 더 의미있는 선택
→ "ing" 같은 어미가 의미 있는 단위로 인식됨
→ "th"는 병합 안 되는 경우도 있음 (InfoGain 낮음)
```

### 2.4 WordPiece 어휘 구성

```
mBERT 어휘 크기: ~110,000개

특징:
- 다국어 포함 (100+ 언어)
- 특수 토큰: [CLS], [SEP], [PAD], [UNK]
- 서브워드: 기본 문자 + ##으로 시작하는 서브워드

구성 예:
{'[CLS]', '[SEP]', '[PAD]'}
{'a', 'b', 'c', ...}
{'##a', '##b', ...}
{'play', 'ing', 'said', ...}
{'##ing', '##ed', ...}
```

---

## 📊 2가지 모델 개념 비교

| 측면 | BPE | WordPiece |
|------|-----|-----------|
| **선택 기준** | 빈도 | 확률 |
| **알고리즘** | Greedy 병합 | InfoGain 최대화 |
| **사전 토크나이저** | 불필요 | **필수** |
| **공백 처리** | Ġ (특수 문자) | [없음] |
| **서브워드 표시** | 없음 | ## (접두사) |
| **의미 기반** | 아니오 | **예** |
| **대표 모델** | GPT-2/3/4 | BERT, mBERT |
| **어휘 크기** | 50K | 110K |
| **처리 속도** | 빠름 | 빠름 |
| **한글 처리** | 약함 | 중간 |

---

## 🎯 이해해야 할 핵심

### BPE (GPT-2)
- ✅ "가장 자주 나오는 쌍을 계속 합친다" = 탐욕적 선택
- ✅ 의미를 무시하고 순수 빈도만 고려
- ✅ 빈도 기반이므로 한글 처리 약함

### WordPiece (mBERT)
- ✅ "확률이 가장 높아지는 조합을 선택" = 확률 기반
- ✅ ## 접두사로 서브워드 위치 명시
- ✅ 사전 토크나이저 필수 (띄어쓰기 기준)
- ✅ 의미 있는 단위 추출 가능

---

## 🔧 다음 단계

**02_practice.ipynb에서:**
1. 이 2가지 모델을 **실제 코드로 로드**
2. 같은 텍스트(영문, 한글, 혼합)로 **직접 토크나이제이션**
3. 결과 **비교 및 분석**
4. 시각화로 **패턴 발견**

이론으로 배운 개념이 실제 코드에서 어떻게 나타나는지 확인합니다!